In [ ]:
import cv2
import numpy as np
import sys

# ============================================
# Funções para adicionar ruído
# ============================================

def add_gaussian_noise(image, mean=0, std=25):
    """Adiciona ruído Gaussiano à imagem."""
    noise = np.random.normal(mean, std, image.shape).astype(np.uint8)
    noisy_image = cv2.add(image, noise)
    return noisy_image

def add_salt_and_pepper_noise(image, noise_ratio=0.02):
    """Adiciona ruído Salt-and-Pepper à imagem."""
    noisy_image = image.copy()
    h, w, c = noisy_image.shape
    noisy_pixels = int(h * w * noise_ratio)

    for _ in range(noisy_pixels):
        row, col = np.random.randint(0, h), np.random.randint(0, w)
        if np.random.rand() < 0.5:
            noisy_image[row, col] = [0, 0, 0]       # pimenta (preto)
        else:
            noisy_image[row, col] = [255, 255, 255] # sal (branco)

    return noisy_image

# ============================================
# Parâmetros
# ============================================
filename = 'foto_grupo_2.png'   # Nome da imagem de entrada (ajuste conforme necessário)
kernels = [3, 5, 11, 29]         # Tamanhos dos kernels para os filtros

# ============================================
# Carregar imagem original
# ============================================
src = cv2.imread(filename)
if src is None:
    print(f"Erro: não foi possível carregar a imagem '{filename}'. Verifique o caminho.")
    sys.exit(1)

# ============================================
# Criar e salvar versões com ruído
# ============================================
print("Aplicando ruídos à imagem original e salvando...")
gaussian_noisy = add_gaussian_noise(src, mean=0, std=25)
sp_noisy = add_salt_and_pepper_noise(src, noise_ratio=0.02)

# Salvar as imagens ruidosas
cv2.imwrite("ruido_gaussiano.jpg", gaussian_noisy)
cv2.imwrite("ruido_salt_pepper.jpg", sp_noisy)
print("Imagens ruidosas salvas: ruido_gaussiano.jpg, ruido_salt_pepper.jpg")

# (Opcional) Visualizar as imagens com ruído
cv2.imshow("Original", src)
cv2.imshow("Ruido Gaussiano", gaussian_noisy)
cv2.imshow("Ruido Salt-and-Pepper", sp_noisy)
cv2.waitKey(0)
cv2.destroyAllWindows()

# ============================================
# Dicionário com todas as imagens a serem filtradas
# ============================================
images_to_filter = {
    'original': src,          # imagem original sem ruído
    'gauss': gaussian_noisy,  # com ruído Gaussiano
    'sp': sp_noisy            # com ruído salt-and-pepper
}

# ============================================
# Aplicar filtros a cada imagem e salvar
# ============================================
print("Iniciando filtragem das imagens (original e ruidosas)...")

for img_name, img in images_to_filter.items():
    for k in kernels:
        # ----- Filtro de média -----
        kernel = np.ones((k, k), np.float32) / (k * k)
        media = cv2.filter2D(img, -1, kernel)
        cv2.imwrite(f"media_{img_name}_{k}x{k}.jpg", media)

        # ----- Filtro Gaussiano -----
        gaussian = cv2.GaussianBlur(img, (k, k), 0)
        cv2.imwrite(f"gaussian_{img_name}_{k}x{k}.jpg", gaussian)

        # ----- Filtro Mediana -----
        median = cv2.medianBlur(img, k)
        cv2.imwrite(f"median_{img_name}_{k}x{k}.jpg", median)

        # ----- Filtro Bilateral -----
        bilateral = cv2.bilateralFilter(img, k, k*2, k//2)
        cv2.imwrite(f"bilateral_{img_name}_{k}x{k}.jpg", bilateral)

print("Processamento concluído. Todas as imagens foram salvas.")